# 订单延迟数据

为了得到更真实的回测结果，必须考虑延迟。因此，需要同时采集带时间戳的行情数据和订单数据，用来测量订单延迟。最好的方式是采集自己的订单延迟：可以基于实盘交易记录，也可以定期提交不会成交的订单并随后撤单，用于记录延迟。

如果没有这类数据，或者只是想建立一个目标延迟，就需要人工生成订单延迟。可以根据行情延迟、成交量、事件数量等因素建模。本教程演示一种简单方法：使用乘数和偏移量，从行情延迟生成订单延迟。

首先加载行情数据。

In [1]:
import numpy as np

data = np.load('btcusdt_20200201.npz')['data']
data

array([(3758096386, 1580515202342000000, 1580515202497052000, 9364.51, 1.197, 0, 0, 0.),
       (3758096386, 1580515202342000000, 1580515202497346000, 9365.67, 0.02 , 0, 0, 0.),
       (3758096386, 1580515202342000000, 1580515202497352000, 9365.86, 0.01 , 0, 0, 0.),
       ...,
       (3489660929, 1580601599836000000, 1580601599962961000, 9351.47, 3.914, 0, 0, 0.),
       (3489660929, 1580601599836000000, 1580601599963461000, 9397.78, 0.1  , 0, 0, 0.),
       (3489660929, 1580601599848000000, 1580601599973647000, 9348.14, 3.98 , 0, 0, 0.)],
      dtype=[('ev', '<i8'), ('exch_ts', '<i8'), ('local_ts', '<i8'), ('px', '<f8'), ('qty', '<f8'), ('order_id', '<u8'), ('ival', '<i8'), ('fval', '<f8')])

为了便于处理，将其转换为 DataFrame。

In [2]:
import polars as pl

df = pl.DataFrame(data)
df

ev,exch_ts,local_ts,px,qty,order_id,ival,fval
i64,i64,i64,f64,f64,u64,i64,f64
3758096386,1580515202342000000,1580515202497052000,9364.51,1.197,0,0,0.0
3758096386,1580515202342000000,1580515202497346000,9365.67,0.02,0,0,0.0
3758096386,1580515202342000000,1580515202497352000,9365.86,0.01,0,0,0.0
3758096386,1580515202342000000,1580515202497357000,9366.36,0.002,0,0,0.0
3758096386,1580515202342000000,1580515202497363000,9366.36,0.003,0,0,0.0
…,…,…,…,…,…,…,…
3489660929,1580601599812000000,1580601599944404000,9397.79,0.0,0,0,0.0
3489660929,1580601599826000000,1580601599952176000,9354.8,4.07,0,0,0.0
3489660929,1580601599836000000,1580601599962961000,9351.47,3.914,0,0,0.0


只选择同时具有有效交易所时间戳和有效本地时间戳的事件，用来计算行情延迟。

In [3]:
from hftbacktest import EXCH_EVENT, LOCAL_EVENT

df = df.filter((pl.col('ev') & EXCH_EVENT == EXCH_EVENT) & (pl.col('ev') & LOCAL_EVENT == LOCAL_EVENT))

通过重采样到约 1 秒间隔，减少数据行数。

In [4]:
df = df.with_columns(
    pl.col('local_ts').alias('ts')
).group_by_dynamic(
    'ts', every='1000000000i'
).agg(
    pl.col('exch_ts').last(),
    pl.col('local_ts').last()
).drop('ts')

df

exch_ts,local_ts
i64,i64
1580515202843000000,1580515202979365000
1580515203551000000,1580515203943566000
1580515203789000000,1580515204875639000
1580515204127000000,1580515205962135000
1580515204738000000,1580515206983780000
…,…
1580601595869000000,1580601595997115000
1580601596865000000,1580601596994060000
1580601597864000000,1580601597987786000


再转换回结构化 NumPy 数组。

In [5]:
data = df.to_numpy(structured=True)
data

array([(1580515202843000000, 1580515202979365000),
       (1580515203551000000, 1580515203943566000),
       (1580515203789000000, 1580515204875639000), ...,
       (1580601597864000000, 1580601597987786000),
       (1580601598870000000, 1580601598997068000),
       (1580601599848000000, 1580601599973647000)],
      dtype=[('exch_ts', '<i8'), ('local_ts', '<i8')])

生成订单延迟。订单延迟由两部分组成：订单请求到达交易所撮合引擎前的延迟，以及回报返回本地前的延迟。订单延迟不等于行情延迟，也不一定与行情延迟成比例。为简化处理，这里用乘数和偏移量，把订单延迟建模为与行情延迟成比例。

In [6]:
mul_entry = 4
offset_entry = 0

mul_resp = 3
offset_resp = 0

order_latency = np.zeros(len(data), dtype=[('req_ts', 'i8'), ('exch_ts', 'i8'), ('resp_ts', 'i8'), ('_padding', 'i8')])
for i, (exch_ts, local_ts) in enumerate(data):
    feed_latency = local_ts - exch_ts
    order_entry_latency = mul_entry * feed_latency + offset_entry
    order_resp_latency = mul_resp * feed_latency + offset_resp

    req_ts = local_ts
    order_exch_ts = req_ts + order_entry_latency
    resp_ts = order_exch_ts + order_resp_latency
    
    order_latency[i] = (req_ts, order_exch_ts, resp_ts, 0)
    
order_latency

array([(1580515202979365000, 1580515203524825000, 1580515203933920000, 0),
       (1580515203943566000, 1580515205513830000, 1580515206691528000, 0),
       (1580515204875639000, 1580515209222195000, 1580515212482112000, 0),
       ...,
       (1580601597987786000, 1580601598482930000, 1580601598854288000, 0),
       (1580601598997068000, 1580601599505340000, 1580601599886544000, 0),
       (1580601599973647000, 1580601600476235000, 1580601600853176000, 0)],
      dtype=[('req_ts', '<i8'), ('exch_ts', '<i8'), ('resp_ts', '<i8'), ('_padding', '<i8')])

In [7]:
df_order_latency = pl.DataFrame(order_latency)
df_order_latency

req_ts,exch_ts,resp_ts,_padding
i64,i64,i64,i64
1580515202979365000,1580515203524825000,1580515203933920000,0
1580515203943566000,1580515205513830000,1580515206691528000,0
1580515204875639000,1580515209222195000,1580515212482112000,0
1580515205962135000,1580515213302675000,1580515218808080000,0
1580515206983780000,1580515215966900000,1580515222704240000,0
…,…,…,…
1580601595997115000,1580601596509575000,1580601596893920000,0
1580601596994060000,1580601597510300000,1580601597897480000,0
1580601597987786000,1580601598482930000,1580601598854288000,0


检查延迟中是否存在无效的负值。

In [8]:
order_entry_latency = df_order_latency['exch_ts'] - df_order_latency['req_ts']
order_resp_latency = df_order_latency['resp_ts'] - df_order_latency['exch_ts']

In [9]:
(order_entry_latency <= 0).sum()

0

In [10]:
(order_resp_latency <= 0).sum()

0

这里把整个过程封装为一个使用 `njit` 的方法，以提升速度。

In [11]:
import numpy as np
from numba import njit
import polars as pl
from hftbacktest import LOCAL_EVENT, EXCH_EVENT

@njit
def generate_order_latency_nb(data, order_latency, mul_entry, offset_entry, mul_resp, offset_resp):
    for i in range(len(data)):
        exch_ts = data[i].exch_ts
        local_ts = data[i].local_ts
        feed_latency = local_ts - exch_ts
        order_entry_latency = mul_entry * feed_latency + offset_entry
        order_resp_latency = mul_resp * feed_latency + offset_resp

        req_ts = local_ts
        order_exch_ts = req_ts + order_entry_latency
        resp_ts = order_exch_ts + order_resp_latency

        order_latency[i].req_ts = req_ts
        order_latency[i].exch_ts = order_exch_ts
        order_latency[i].resp_ts = resp_ts

def generate_order_latency(feed_file, output_file = None, mul_entry = 1, offset_entry = 0, mul_resp = 1, offset_resp = 0):
    data = np.load(feed_file)['data']
    df = pl.DataFrame(data)
    
    df = df.filter(
        (pl.col('ev') & EXCH_EVENT == EXCH_EVENT) & (pl.col('ev') & LOCAL_EVENT == LOCAL_EVENT)
    ).with_columns(
        pl.col('local_ts').alias('ts')
    ).group_by_dynamic(
        'ts', every='1000000000i'
    ).agg(
        pl.col('exch_ts').last(),
        pl.col('local_ts').last()
    ).drop('ts')
    
    data = df.to_numpy(structured=True)

    order_latency = np.zeros(len(data), dtype=[('req_ts', 'i8'), ('exch_ts', 'i8'), ('resp_ts', 'i8'), ('_padding', 'i8')])
    generate_order_latency_nb(data, order_latency, mul_entry, offset_entry, mul_resp, offset_resp)

    if output_file is not None:
        np.savez_compressed(output_file, data=order_latency)

    return order_latency

In [12]:
order_latency = generate_order_latency('btcusdt_20200201.npz', output_file='feed_latency_20200201.npz', mul_entry=4, mul_resp=3)